<img src = "https://drive.google.com/uc?export=view&id=1QqjbbEZ1w7xoawV020Jj_R46PKRi6A_e" alt = "Encabezado MLDS" width = "100%">  </img>

# **Tarea 1: Clasificación lineal**
---

En esta tarea deberá entrenar modelos de clasificación con regresión logística para el [conjunto de datos de vinos Wine](https://archive.ics.uci.edu/ml/machine-learning-databases/wine/wine.data) del repositorio de la *UCI* usando *Keras* con el back end de *TensorFlow*.

Ejecute las siguientes celdas para conectarse a UNCode:

# 📘 Tarea 1 · Versión explicada

Copia del notebook resuelto con una explicación antes de cada celda de código. El código y las salidas son los de la ejecución real que obtuvo **100 % en UNCode**, con las 12 pruebas en 100. El original sin explicaciones es `v2_M5U1_Tarea_1_Resuelto.ipynb`.

**Cómo leer cada explicación**
- **📘 Objetivo de la celda:** para qué existe la celda dentro del flujo.
- **Qué hace cada parte del código:** cada instrucción relevante y su función.
- **Punto clave:** el detalle que conviene recordar, cuando lo hay.
- **🔎 Resultado:** la lectura de la salida obtenida.

| | |
|---|---|
| **Problema** | Clasificación binaria entre dos tipos de vino del dataset *Wine* |
| **Modelo** | Regresión logística: una neurona `Dense(1)` con sigmoide, 14 parámetros |
| **Datos** | 83 muestras de entrenamiento y 36 de prueba, con 13 variables químicas; 50 frente a 33 por clase en entrenamiento |
| **Técnica central** | Entropía cruzada **ponderada** para compensar el desbalance |
| **Resultado** | 98,8 % de accuracy en entrenamiento |

```
Wine, clases 2 y 3 → 0 y 1
   → Punto 1 class_weights    pesos por clase           → [0,83 , 1,258]
   → Punto 2 get_loss         entropía cruzada ponderada
   → Punto 3 optimizer        SGD, Adam o RMSprop
   → Punto 4 get_model        Dense 1 con sigmoide, compilado
   → Punto 5 train_model      model.fit, 10 épocas
   → Punto 6 model_predict    probabilidades de la clase 1
```

### 📘 Instalación del calificador

**Objetivo de la celda.** Preparar el entorno para que el notebook pueda enviarse a UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `%%capture` | Oculta la salida de la celda |
| `!pip install rlxcrypt` | Instala la librería que descifra el módulo del calificador |
| `!wget ... session.pye` | Descarga el módulo cifrado del calificador |

In [1]:
%%capture
!pip install rlxcrypt
!wget --no-cache -O session.pye -q https://raw.githubusercontent.com/JuezUN/INGInious/master/external%20libs/session.pye

### 📘 Inicio de sesión en UNCode

**Objetivo de la celda.** Crear el objeto `grader`, que ejecuta las pruebas y registra la entrega de esta tarea.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import rlxcrypt, session` | Carga el descifrador y el módulo del calificador |
| `session.LoginSequence('DL-GroupMLDS-5-2026-2@...')` | Abre la sesión del curso y de la tarea; pide usuario y contraseña |

> **Punto clave.** Todas las celdas `grader.` dependen de esta. La contraseña no queda guardada.

In [2]:
import rlxcrypt
import session

grader = session.LoginSequence('DL-GroupMLDS-5-2026-2@cd16b03a-f1d7-4cc0-9a82-61ed5c8d030d')

Please enter your UNCode username: nremolina
Please enter your password: ··········


> Al finalizar, podrás revisar en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/cd16b03a-f1d7-4cc0-9a82-61ed5c8d030d)

Ejecute la siguiente celda para importar y configurar las librerías usadas :

### 📘 Librerías

**Objetivo de la celda.** Importar las herramientas del notebook y fijar la semilla de NumPy.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import numpy as np` | Arreglos numéricos |
| `import tensorflow as tf, keras` | Construcción y entrenamiento del modelo |
| `import matplotlib.pyplot as plt` | Gráficas |
| `seed(1)` | Fija el azar de NumPy |
| `warnings.filterwarnings('ignore')` | Oculta avisos que no afectan el resultado |

In [16]:
# Librerías de utilidad para manipulación y visualización de datos.
from numbers import Number
import numpy as np
import tensorflow as tf
import keras
import matplotlib as mpl
import matplotlib.pyplot as plt

from numpy.random import seed
seed(1)

# Ignorar warnings.
import warnings
warnings.filterwarnings('ignore')

### 📘 Versiones del entorno

**Objetivo de la celda.** Registrar con qué versiones se ejecutó el notebook.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `!python --version` | Muestra la versión de Python |
| `tf.__version__, keras.__version__` | Muestran las versiones de TensorFlow y Keras |

In [17]:
# Versiones de las librerías usadas.
!python --version
print('Tensorflow', tf.__version__)
print('Keras', keras.__version__)

Python 3.13.16
Tensorflow 2.21.0
Keras 3.13.2


#### 🔎 Resultado

Python 3.13.16, TensorFlow 2.21.0 y Keras 3.13.2, más nuevas que las del enunciado: Python 3.11.12, TensorFlow 2.18.0 y Keras 3.8.0.

Esta actividad se realizó con las siguientes versiones:
*  Python 3.11.12
*  Tensorflow 2.18.0
*  Keras 3.8.0

## **Cargar los datos**
---
En el conjunto *Wine* las características de entrada corresponden a diferentes atributos del vino. El conjunto de datos contiene 178 ejemplos sobre los que se tiene la siguiente información :

* Alcohol
* Malic acid
* Ash
* Alcalinity of ash
* Magnesium
* Total phenols
* Flavanoids
* Nonflavanoid phenols
* Proanthocyanins
* Color intensity
* Hue
* OD280/OD315 of diluted wines
* Proline

Hay tres clases de vinos diferentes. `class_0` el cual tiene 59 muestras, `class_1` el cual tiene 71 muestras y `class_2` el cual tiene 48 muestras.

Como en cualquier experimento de _machine learning_, vamos a empezar cargando el conjunto de datos, haciendo particiones de entrenamiento y prueba, y para efectos de esta tarea, nos vamos a quedar solo con dos clases (`class_1` y `class_2`) para hacer clasificación:

### 📘 Descarga de los datos

**Objetivo de la celda.** Traer el archivo `wine.data` desde el repositorio del curso.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `!wget ... wine.data` | Descarga el archivo y lo guarda con ese nombre |

In [18]:
!wget --no-cache -O wine.data -q  https://raw.githubusercontent.com/mindlab-unal/mlds5-datasets/main/u1/taller/wine.data?raw=true

### 📘 Preparación de los datos

**Objetivo de la celda.** Quedarse con dos tipos de vino, escalar las variables y separar entrenamiento y prueba.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `pd.read_csv('wine.data', header=None)` | Lee el archivo; no trae encabezados |
| `data.iloc[:, 1:] y data.iloc[:, 0]` | Separa las 13 variables de la etiqueta, que está en la primera columna |
| `np.where(...)` | Filtra solo los vinos tipo 2 y 3 |
| `... - 2` | Recodifica las etiquetas a 0 y 1 |
| `MinMaxScaler((0, 1)).fit_transform(X)` | Lleva cada variable al rango de 0 a 1 |
| `train_test_split(test_size=0.3, stratify=y, random_state=42)` | Reserva el 30 % para prueba, manteniendo la proporción de clases |

> **Punto clave.** El escalador se ajusta con todos los datos antes de dividir. Es una pequeña fuga de información: lo correcto sería ajustarlo solo con el entrenamiento.

In [19]:
import pandas as pd
from sklearn import preprocessing
from sklearn import model_selection

# Leer el archivo que contiene los datos:
data =  pd.read_csv('wine.data', sep=",", header=None)
# La etiqueta está consignada en la primera columna:
X_all = np.array(data.iloc[:,1:])
y_all = np.array(data.iloc[:,0])
# Nos quedamos con la clase 2 y 3, y ajustamos las etiquetas para que queden
# como 0 y 1:
X = X_all[np.where((y_all==2)|(y_all==3))]
y = y_all[np.where((y_all==2)|(y_all==3))]-2
# Re-escalamos los datos
scaler = preprocessing.MinMaxScaler((0, 1))
X = scaler.fit_transform(X)
# Y hacemos partición en entrenamiento y prueba
X_train, X_test, y_train, y_test = model_selection.train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)

Verifiquemos el tamaño de las particiones:

### 📘 Tamaño de las particiones

**Objetivo de la celda.** Comprobar cuántas muestras y variables tiene cada conjunto.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `X_train.shape[0], X_test.shape[0]` | Número de muestras de entrenamiento y de prueba |
| `X_train.shape[1]` | Número de variables |

In [20]:
print("Número de muestras de entrenamiento =", X_train.shape[0])
print("Número de muestras de prueba =", X_test.shape[0])
print("Número de características del conjunto de datos =", X_train.shape[1])

Número de muestras de entrenamiento = 83
Número de muestras de prueba = 36
Número de características del conjunto de datos = 13


#### 🔎 Resultado

83 muestras de entrenamiento, 36 de prueba y 13 variables. El enunciado dice luego «122 muestras», pero son 119.

**Salida esperada**

```python
Número de muestras de entrenamiento = 83
Número de muestras de prueba = 36
Número de características del conjunto de datos = 13
```

Tenemos entonces 122 muestras con 13 _features_ para trabajar. Ahora veamos cuántas muestras hay por cada clase:

### 📘 Muestras por clase

**Objetivo de la celda.** Medir el desbalance entre las dos clases.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `X_train[np.where(y_train==0)].shape[0]` | Cuenta las muestras de la clase 0 en entrenamiento |
| `X_train[np.where(y_train==1)].shape[0]` | Cuenta las de la clase 1; lo mismo se repite para la prueba |

In [21]:
print("Número de muestras de la clase 2 en entrenamiento =", X_train[np.where(y_train==0)].shape[0])
print("Número de muestras de la clase 3 en entrenamiento =", X_train[np.where(y_train==1)].shape[0])
print("Número de muestras de la clase 2 en prueba =", X_test[np.where(y_test==0)].shape[0])
print("Número de muestras de la clase 3 en prueba =", X_test[np.where(y_test==1)].shape[0])

Número de muestras de la clase 2 en entrenamiento = 50
Número de muestras de la clase 3 en entrenamiento = 33
Número de muestras de la clase 2 en prueba = 21
Número de muestras de la clase 3 en prueba = 15


#### 🔎 Resultado

50 frente a 33 en entrenamiento y 21 frente a 15 en prueba: la clase 1 tiene un 34 % menos de muestras. Ese desbalance es el que compensa la pérdida ponderada.

**Salida esperada**
```python
Número de muestras de la clase 2 en entrenamiento = 50
Número de muestras de la clase 3 en entrenamiento = 33
Número de muestras de la clase 2 en prueba = 21
Número de muestras de la clase 3 en prueba = 15
```

Como puede ver, el conjunto de datos está desbalanceado. Vamos entonces a implementar un modelo que compense este desbalance desde la función de pérdida.

## **Modelo de regresión logística con _class_weight_**
---
Una vez se dispone de un conjunto de datos preparado para el entrenamiento, se declara el algoritmos de aprendizaje computacional. En nuestro caso queremos predecir el valor de una variable categórica, es decir, realizar un modelo para **clasificación**.

Sin embargo, el conjunto de datos no está balanceado. Cuando esto sucede, podemos compensar el desbalance dándole más importancia a la clase menos presente. Darle más importancia a una clase se logra asignando un peso por cada clase dentro de la función de pérdida del modelo.

Supongamo que tenemos un problema desbalanceado de clasificación binario con etiquetas $0$ y $1$. Supongamos que $n_0$ es el número de elementos de la clase $0$ y $n_1$ es el número de elementos de la clase $1$. Una elección convencional sobre los pesos que se le deben asignar a cada clase es :

$$w_0=\dfrac{n_0+n_1}{2n_0},$$

$$w_1=\dfrac{n_0+n_1}{2n_1}.$$

Y estos pesos se incorporan a la función de pérdida de la siguiente forma :

$$\mathcal{L}(\vec{w})=-\frac{1}{N}\sum_{i=1}^{N}[w_1 y_i\log(\tilde{y}_i)+w_0(1-y_i)\log(1-\tilde{y}_i)],$$

¿Cómo funciona? Supongamos que hay 100 muestras de la clase $0$ y 50 de la clase $1$. Es decir, $n_0=100$ y $n_1=50$. Entonces $w_0=0.75$ y $w_1=1.5$. Así, el peso de la clase $1$ es el doble del peso de la clase $0$. Cuando el modelo no clasifica bien una muestra de la clase $1$, la penalidad se multiplica por $w_1$. Es decir, el modelo entiende que es más grave equivocarse con los datos de la clase $1$, y de esa manera compensa el que sean menos muestras que las de la clase $0$, tratanto de previnir cualquier tipo de sesgo en el modelo final.



> **La tarea es incremental, por lo tanto es recomendable resolver los puntos en orden**

## **1. Calcular el peso de cada clase**
---

Complete la función **`class_weights`** para que calcule los pesos que el modelo tiene que darle a cada clase según el desbalance de los datos.

**Entrada** :

* **`y`**: un `numpy.ndarray` de tamaño `(m,)`; es decir, el vector de etiquetas de los datos de entrenamiento. $m$ el número de muestras del conjunto de entrenamiento.

**Salida** :

* **`weights_list`** : `list`, una lista con los pesos (tipo `float`) de cada clase, en orden.

### 📘 Punto 1 · class_weights

**Objetivo de la celda.** Calcular cuánto debe pesar cada clase en la pérdida según cuántas muestras tiene.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import numpy, tensorflow, keras` | El calificador ejecuta cada función aislada y no ve los `import` del notebook; por eso se repiten dentro. |
| `n_0 = np.sum(y == 0)` | Cuenta las muestras de la clase 0 |
| `n_1 = np.sum(y == 1)` | Cuenta las muestras de la clase 1 |
| `w_0 = float((n_0 + n_1) / (2 * n_0))` | Peso de la clase 0: total entre el doble de sus muestras |
| `w_1 = float((n_0 + n_1) / (2 * n_1))` | Peso de la clase 1, la minoritaria, que queda mayor |
| `weights_list = [w_0, w_1]` | Devuelve los pesos en orden |

> **Punto clave.** Con esta fórmula el peso promedio por muestra es exactamente 1, así que la pérdida ponderada queda en la misma escala que la normal.

In [46]:
# FUNCIÓN CALIFICADA class_weights:

def class_weights(y):
    import numpy as np
    import tensorflow as tf
    import keras
    # Número de muestras por clase
    n_0 = np.sum(y == 0)
    n_1 = np.sum(y == 1)
    # w_k = (n_0 + n_1) / (2 * n_k)
    w_0 = float((n_0 + n_1) / (2 * n_0))
    w_1 = float((n_0 + n_1) / (2 * n_1))
    weights_list = [w_0, w_1]
    return weights_list

### 📘 Prueba local del punto 1

**Objetivo de la celda.** Calcular los pesos con el entrenamiento y compararlos con los esperados.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `class_weights(y_train)` | Calcula los pesos con las 83 etiquetas de entrenamiento |
| `np.round(..., 5)` | Redondea a 5 decimales para comparar |

In [47]:
#TEST_CELL

class_weights_list = class_weights(y_train)
print(np.round(class_weights_list, 5))

[0.83    1.25758]


#### 🔎 Resultado

`[0.83 1.25758]`: igual al esperado. Con 50 y 33 muestras, `w_0 = 83/100` y `w_1 = 83/66`. Un error en la clase 1 pesa 1,52 veces más.

**Salida esperada**:

```python
[0.83    1.25758]
```

### **Evaluar código**

### 📘 Pruebas del punto 1

**Objetivo de la celda.** Verificar `class_weights` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 1_1` y `Test 1_2`.

In [48]:
grader.run_test("Test 1_1", globals())

Test 1_1


In [49]:
grader.run_test("Test 1_2", globals())

Test 1_2


## **2. Binary cross-entropy**
---
Ahora, complete la función **`get_loss`**. Esta debera dar como resultado una función que retorne el valor de la entropia cruzada, teniendo en cuenta los pesos de cada clase.

**Entrada**:

* **`class_weights`** : `list`, una lista con los pesos asociados a cada clase.

**Salida**:

* **`loss`** : `function`, una función que calcula la entropia cruzada ponderada y recibe unicamente los paramteros **`y_true`** y **`y_pred`**

### 📘 Punto 2 · get_loss

**Objetivo de la celda.** Construir la entropía cruzada binaria ponderada como una función que Keras pueda usar como pérdida.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `def get_loss(class_weights)` | Recibe los pesos; la plantilla traía otra firma que no coincide con las pruebas |
| `w_0, w_1 = class_weights` | Separa los dos pesos |
| `def loss(y_true, y_pred)` | Función interna con la firma que exige Keras |
| `keras.ops.cast(y_true, y_pred.dtype)` | Iguala el tipo de dato de las etiquetas al de las predicciones |
| `keras.ops.clip(y_pred, eps, 1 - eps)` | Evita `log(0)` recortando las predicciones |
| `w_1 * y_true * log(y_pred)` | Término de la clase 1, multiplicado por su peso |
| `w_0 * (1 - y_true) * log(1 - y_pred)` | Término de la clase 0, multiplicado por su peso |
| `-keras.ops.mean(...)` | Promedia sobre las muestras y cambia el signo |
| `return loss` | Devuelve la función, no un número |

> **Punto clave.** Es un *closure*: `loss` recuerda `w_0` y `w_1` aunque solo reciba `y_true` y `y_pred`. Usar `keras.ops` y respetar el tipo de `y_pred` fue necesario: una versión con `tf.cast` a `float32` obtuvo 10.

In [70]:
# FUNCIÓN CALIFICADA weighted_bce:
def get_loss(class_weights):
    import keras
    w_0, w_1 = class_weights
    def loss(y_true, y_pred):
        # Igualar el tipo de dato de y_true al de y_pred
        y_true = keras.ops.cast(y_true, y_pred.dtype)
        # Evitar log(0)
        eps = keras.backend.epsilon()
        y_pred = keras.ops.clip(y_pred, eps, 1 - eps)
        w_bce = -keras.ops.mean(w_1 * y_true * keras.ops.log(y_pred)
                                + w_0 * (1 - y_true) * keras.ops.log(1 - y_pred))
        return w_bce
    return loss

Use las siguientes celdas para probar su modelo:

### 📘 Prueba local del punto 2

**Objetivo de la celda.** Evaluar la pérdida con predicciones de 0,5 para todas las muestras.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `np.expand_dims(y_train, axis=-1)` | Da a las etiquetas forma de columna, `(83, 1)` |
| `np.ones(...) * 0.5` | Predicción neutra de 0,5 para cada muestra |
| `loss(y_true, y_pred).numpy()` | Calcula la pérdida y la convierte a número |

In [71]:
#TEST_CELL
class_weights_example = class_weights(y_train)
Y_t = np.expand_dims(y_train, axis=-1)
y_true = Y_t
y_pred = np.ones([Y_t.shape[0],1]) * 0.5
loss = get_loss(class_weights_example)
print("weighted_bce =", loss(y_true,y_pred).numpy())

weighted_bce = 0.6931471805599454


#### 🔎 Resultado

`0.6931471805599454`, que es ln 2. Con predicciones de 0,5 la pérdida vale lo mismo para todas las muestras y, como el peso promedio es 1, coincide con la entropía cruzada sin ponderar.

**Salida esperada**:

```python
weighted_bce = 0.6931471

```

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Pista 1</b></font>
</summary>


* Utilice la función [**`keras.ops.mean`**] de la guia de _keras_.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Pista 2</b></font>
</summary>


* El uso de una función de pérdida de entropía cruzada ponderada implica el uso de los pesos de cada clase para hallar su valor.

<details>    
<summary>
    <font size="3" color="darkgreen"><b>Pista 3</b></font>
</summary>


* Tenga en cuenta que la salida de esta función debe ser otra función


### **Evaluar código**

### 📘 Pruebas del punto 2

**Objetivo de la celda.** Verificar `get_loss` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 2_1` y `Test 2_2`.

In [72]:
grader.run_test("Test 2_1", globals())

Test 2_1


In [73]:
grader.run_test("Test 2_2", globals())

Test 2_2


## **3. Optimizador**
---
Complete la función **`optimizer`** para que devuelva un optimizador válido. Deberá específicar el tipo de optimizador y la tasa de aprendizaje que usaremos en el entrenamiento.

**Entrada** :

* **`type_opt`** : `str`, que puede tomar valores entre: **`SGD`**, **`Adam`**, **`RMSprop`**.
* **`learning_rate`** : `float`, correspondiente a la tasa de aprendizaje.

**Salida** :

* **`opt`** : El optimizador con tasa de aprendizaje definida, un objeto tipo `keras.optimizers`.

### 📘 Punto 3 · optimizer

**Objetivo de la celda.** Devolver el optimizador pedido con la tasa de aprendizaje indicada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import numpy, tensorflow, keras` | El calificador ejecuta cada función aislada y no ve los `import` del notebook; por eso se repiten dentro. |
| `if type_opt == 'SGD'` | Descenso de gradiente estocástico |
| `elif type_opt == 'Adam'` | Adam, con tasa adaptativa por parámetro |
| `elif type_opt == 'RMSprop'` | RMSprop, que normaliza por el promedio de gradientes recientes |
| `raise ValueError(...)` | Avisa si el nombre no es uno de los tres permitidos |

In [74]:
# FUNCIÓN CALIFICADA optimizer:
def optimizer(type_opt, learning_rate):
    import numpy as np
    import tensorflow as tf
    import keras
    if type_opt == 'SGD':
        opt = keras.optimizers.SGD(learning_rate=learning_rate)
    elif type_opt == 'Adam':
        opt = keras.optimizers.Adam(learning_rate=learning_rate)
    elif type_opt == 'RMSprop':
        opt = keras.optimizers.RMSprop(learning_rate=learning_rate)
    else:
        raise ValueError("type_opt debe ser 'SGD', 'Adam' o 'RMSprop'")
    return opt

Use las siguientes celdas para probar su modelo:

### 📘 Prueba local del punto 3

**Objetivo de la celda.** Crear un optimizador Adam con tasa 0,5.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `optimizer('Adam', 0.5)` | Devuelve el objeto optimizador |

In [75]:
#TEST_CELL

type_opt_test = 'Adam'
learning_rate_test = 0.5
optimizer(type_opt_test,learning_rate_test)

#### 🔎 Resultado

Un objeto Adam. La ruta `keras.src` y la dirección de memoria difieren de la salida esperada solo por la versión de Keras y por la ejecución.

**Salida esperada**:

```
<keras.optimizers.adam.Adam at 0x7f53eddd39d0>
```

### **Evaluar código**

### 📘 Pruebas del punto 3

**Objetivo de la celda.** Verificar `optimizer` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 3_1` y `Test 3_2`.

In [76]:
grader.run_test("Test 3_1", globals())

Test 3_1


In [77]:
grader.run_test("Test 3_2", globals())

Test 3_2


## **4. Defina el modelo**
---
Teniendo una función de coste y un optimizador, podemos generar un modelo. A continuación implementa la función **`get_model`**, en la cual deberá definir el modelo usando _Keras_, considere que este se trata de una regresión logística.

**Entrada** :

* **`n_inputs`** : `int`, el número de vairables que utilizrá el modelo como entrada.
* **`n_neurons`** : `int`, el número de salidas esperadas.
* **`optimizer`** : `keras.optimizers`, el optimizador definido que se usará para minimizar la función de pérdida **`loss_fun`**
* **`loss`**: `function`, Función de perdida a optimizar.

**Salida** :

* **`model`** : `keras.Models.Sequential` Modelo a entrenar.

### 📘 Punto 4 · get_model

**Objetivo de la celda.** Definir la regresión logística en Keras y dejarla compilada.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import numpy, tensorflow, keras` | El calificador ejecuta cada función aislada y no ve los `import` del notebook; por eso se repiten dentro. |
| `keras.models.Sequential([...])` | Modelo de capas en secuencia |
| `keras.layers.Input(shape=(n_inputs,))` | Entrada de 13 variables |
| `keras.layers.Dense(n_neurons, activation='sigmoid')` | Una neurona con sigmoide: devuelve la probabilidad de la clase 1 |
| `model.compile(optimizer=optimizer, loss=loss)` | Asocia el optimizador y la pérdida ponderada |

> **Punto clave.** Una regresión logística es una red neuronal de una sola neurona.

In [78]:
def get_model(n_inputs, n_neurons, optimizer, loss):
    import numpy as np
    import tensorflow as tf
    import keras
    # Regresión logística = una capa densa con activación sigmoide
    model = keras.models.Sequential([
        keras.layers.Input(shape=(n_inputs,)),
        keras.layers.Dense(n_neurons, activation='sigmoid')
    ])
    model.compile(optimizer=optimizer, loss=loss)
    return model

### 📘 Prueba local del punto 4

**Objetivo de la celda.** Construir el modelo con la pérdida ponderada y Adam, y revisar su resumen.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `get_loss(class_weights_example)` | Crea la pérdida con los pesos del entrenamiento |
| `optimizer('Adam', 0.5)` | Crea el optimizador |
| `get_model(13, 1, opt, loss)` | Crea el modelo de 13 entradas y 1 salida |
| `model.summary()` | Muestra la capa y sus parámetros |

In [79]:
#TEST_CELL
class_weights_example = class_weights(y_train)
loss = get_loss(class_weights_example)
opt = optimizer('Adam',0.5)
model = get_model(13, 1, opt, loss)
model.summary()

Model: "sequential_28"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_28 (Dense)                │ (None, 1)              │            14 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14 (56.00 B)

 Trainable params: 14 (56.00 B)

 Non-trainable params: 0 (0.00 B)

#### 🔎 Resultado

Una capa `Dense` con **14 parámetros**: 13 pesos y 1 sesgo. El nombre `sequential_28` solo indica que la celda se ejecutó varias veces.

**Salida esperada**

```
Model: "sequential"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense (Dense)                        │ (None, 1)                   │              14 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘
 Total params: 14 (56.00 B)
 Trainable params: 14 (56.00 B)
 Non-trainable params: 0 (0.00 B)
```

### **Evaluar código**

### 📘 Pruebas del punto 4

**Objetivo de la celda.** Verificar `get_model` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 4_1` y `Test 4_2`.

In [80]:
grader.run_test("Test 4_1", globals())

Test 4_1


In [81]:
grader.run_test("Test 4_2", globals())

Test 4_2


## **5. Entrenamiento del modelo**
---
Una vez definida cada una de las piezas, podemos entrenar el modelo. Complete la función **`train_model`** para que retorne los pesos, del modelo entrenado sobre los arreglos **`X_t`** y **`Y_t`**.

**Entrada** :

* **`model`** : `keras.Models.Sequential` Modelo a entrenar.
* **`epochs`** : `int`, el número de iteraciones durante las cuales se realizará el entrenamiento del modelo.
* **`optimizer`** : `keras.optimizers`, el optimizador definido que se usará para minimizar la función de pérdida **`loss_fun`**
* **`X_t`**: `Tensor`, tensor de tamaño `(m,n)`, correspondiente a la matriz de datos de entrenamiento.
* **`Y_t`**: `Tensor`, tensor de tamaño `(m,1)`, correspondiente a las etiquetas de los datos entrenamiento.
* **`loss`** : `function`, una función que calcula la entropia cruzada ponderada y recibe unicamente los paramteros **`y_true`** y **`y_pred`**




**Salida** :

* **`losses`** : `list` Una lista con los valores de **`loss_fun`** en cada iteración.
* **`weigths`**: `list`, lsita de tensores que contiene los pesos entrandos del mdoelo.
 >nota: Utiliza como referencia la construcción del modelo basado en Keras del taller guiado.


### 📘 Punto 5 · train_model

**Objetivo de la celda.** Entrenar el modelo y devolver la pérdida de cada época y los pesos aprendidos.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `import numpy, tensorflow, keras` | El calificador ejecuta cada función aislada y no ve los `import` del notebook; por eso se repiten dentro. |
| `model.fit(X_t, Y_t, epochs=epochs, verbose=0)` | Entrena sin imprimir; el modelo ya trae optimizador y pérdida desde `get_model` |
| `history.history['loss']` | Lista con la pérdida de cada época |
| `model.get_weights()` | Pesos y sesgo aprendidos |

> **Punto clave.** Se usó `model.fit` y no un bucle manual: `fit` trabaja con lotes de 32 y reporta el promedio de los lotes de cada época, que es lo que reproduce la salida esperada.

In [82]:
def train_model(model, epochs, optimizer, X_t, Y_t, loss):
    import numpy as np
    import tensorflow as tf
    import keras
    # El modelo ya está compilado con el optimizador y la pérdida
    history = model.fit(X_t, Y_t, epochs=epochs, verbose=0)
    losses = history.history['loss']

    return losses, model.get_weights()

### 📘 Prueba local del punto 5

**Objetivo de la celda.** Entrenar 10 épocas desde pesos en cero y comparar pérdidas y pesos con los esperados.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `keras.utils.set_random_seed(1)` | Fija el azar para que el resultado sea siempre el mismo |
| `model.layers[0].set_weights([np.zeros(...), np.array([0])])` | Arranca con todos los pesos y el sesgo en cero |
| `train_model(model, 10, opt, X_t, Y_t, loss)` | Entrena 10 épocas |
| `losses[0], losses[1], weigths[0][0], weigths[1]` | Imprime las dos primeras pérdidas, el primer peso y el sesgo |

In [83]:
#TEST_CELL
keras.utils.set_random_seed(1) ## Fijar semilla para obtener siempre el mismo valor
class_weights_example = class_weights(y_train)
loss = get_loss(class_weights_example)
opt = optimizer('Adam',0.5)

model = get_model(13, 1, opt, loss)
model.layers[0].set_weights([np.zeros([13,1]),np.array([0])])

X_t = X_train
Y_t = np.expand_dims(y_train, axis=-1)
losses, weigths = train_model(model, 10, opt, X_t, Y_t, loss)
print('Epoch', 0, 'loss =', losses[0])
print('Epoch', 1, 'loss =', losses[1])
print("w[0] =", weigths[0][0])
print("b =",weigths[1])

Epoch 0 loss = 0.9399663209915161
Epoch 1 loss = 0.3858067989349365
w[0] = [2.2023726]
b = [-0.09913211]


#### 🔎 Resultado

Pérdidas 0,9399663 y 0,3858068, y `w[0] = 2.2023726`: iguales a las esperadas. El sesgo difiere en la séptima cifra decimal por la versión de TensorFlow.

**Salida esperada**:

```
Epoch 0 loss = 0.9399663209915161
Epoch 1 loss = 0.3858067989349365
w[0] = [2.2023726]
b = [-0.09913202]
```

### **Evaluar código**

### 📘 Pruebas del punto 5

**Objetivo de la celda.** Verificar `train_model` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 5_1` y `Test 5_2`.

In [84]:
grader.run_test("Test 5_1", globals())

Test 5_1


In [85]:
grader.run_test("Test 5_2", globals())

Test 5_2


## **6. Predicciones del modelo**
---
Finalmente, podemos utilizar los datos reservados de la partición de prueba para calcular predicciones y hacer evaluaciones.

Complete la función **`model_predict`** para que retorne las predicciones del modelo entrenado sobre un conjunto de prueba.

**Entrada** :
* **`model`** : `keras.Models.Sequential` Modelo a entrenar.
* **`X_test`** : `Tensor`, tensor de tamaño `(l,n)`, correspondiente a la matriz de datos de prueba.

**Salida** :

* **`y_pred`** : Tensor de tamaño `(l,1)`, con las predicciones de **model** para el conjunto de prueba **`X_test`**.

### 📘 Punto 6 · model_predict

**Objetivo de la celda.** Obtener las probabilidades que el modelo asigna a la clase 1.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `def model_predict(model, X_test)` | La plantilla traía `model_predict(log_reg, b, w, X_test)`; se siguió la firma que usan las pruebas |
| `tf.convert_to_tensor(X_test, dtype=tf.float32)` | Convierte los datos en un tensor |
| `model(...)` | Aplica el modelo y devuelve un tensor de forma `(muestras, 1)` |

> **Punto clave.** `model(X)` devuelve un tensor; `model.predict(X)` devolvería un arreglo de NumPy. La prueba usa `.numpy()` sobre el resultado, así que espera un tensor.

In [86]:
# FUNCIÓN CALIFICADA model_predict:
def model_predict(model, X_test):
    import numpy as np
    import tensorflow as tf
    import keras
    # Probabilidades de pertenecer a la clase 1 (tensor de tamaño (l,1))
    y_pred = model(tf.convert_to_tensor(X_test, dtype=tf.float32))
    return y_pred

Use la siguiente celda para probar su función:

### 📘 Prueba local del punto 6

**Objetivo de la celda.** Entrenar de nuevo, predecir y medir la accuracy.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `train_model(...)` | Entrena con la misma configuración de la prueba anterior |
| `model_predict(model, X_t)` | Predice sobre los datos de entrenamiento |
| `tf.math.round(y_pred)` | Convierte las probabilidades en clases 0 o 1 con umbral de 0,5 |
| `keras.metrics.Accuracy()` | Mide la proporción de aciertos |

In [87]:
keras.utils.set_random_seed(1)
class_weights_example = class_weights(y_train)
loss = get_loss(class_weights_example)
opt = optimizer('Adam',0.5)
model = get_model(13, 1, opt, loss)
model.layers[0].set_weights([np.zeros([13,1]),np.array([0])])

losses, weigths = train_model(model, 10, opt, X_t, Y_t, loss)
y_pred = model_predict(model, X_t)

print("Primeras dos predicciones:\n", y_pred[:2].numpy())
m = keras.metrics.Accuracy()
m.update_state(Y_t, tf.math.round(y_pred))
print("Accuracy sobre X_test después de 10 epochs:", m.result().numpy())

Primeras dos predicciones:
 [[0.9925333]
 [0.0018804]]
Accuracy sobre X_test después de 10 epochs: 0.9879518


#### 🔎 Resultado

Primeras predicciones de 0,9925 y 0,0019, y accuracy de 0,9879518, que es exactamente 82 de 83. Aunque el texto dice «sobre X_test», se calcula sobre el **entrenamiento**; en prueba el modelo acierta 33 de 36, un 91,7 %.

**Salida esperada:**

```
Primeras dos predicciones:
 [[0.9925333]
 [0.0018804]]
Accuracy sobre X_test después de 10 epochs: 0.9879518
```

### **Evaluar código**

### 📘 Pruebas del punto 6

**Objetivo de la celda.** Verificar `model_predict` con el calificador. Esta explicación cubre las dos celdas siguientes, `Test 6_1` y `Test 6_2`.

In [88]:
grader.run_test("Test 6_1", globals())

Test 6_1


In [89]:
grader.run_test("Test 6_2", globals())

Test 6_2


# **Evaluación**

### 📘 Envío oficial

**Objetivo de la celda.** Registrar la entrega en UNCode.

**Qué hace cada parte del código**

| Parte del código | Qué hace |
|---|---|
| `grader.submit_task(globals())` | Vuelve a ejecutar las 12 pruebas en el servidor y guarda la calificación |

In [90]:
grader.submit_task(globals())

Test 1_1
Test 1_2
Test 2_1
Test 2_2
Test 3_1
Test 3_2
Test 4_1
Test 4_2


Test 5_1
Test 5_2
Test 6_1
Test 6_2


#### 🔎 Resultado

**Las 12 pruebas en 100 y envío con 100 %**, según el registro de UNCode. El notebook no conserva el detalle de las notas porque el calificador las muestra como HTML.

Revisa en UNCode tus submissions en la [página de la tarea.](https://uncode.unal.edu.co/course/DL-GroupMLDS-5-2026-2/cd16b03a-f1d7-4cc0-9a82-61ed5c8d030d)

# 📘 Cierre

| Punto | Función | Resultado | Qué fue decisivo |
|---|---|---|---|
| 1 | `class_weights` | 100, 100 | La fórmula `N / (2 n_k)` |
| 2 | `get_loss` | 100, 100 | Firma `get_loss(class_weights)`, closure y `keras.ops` respetando el tipo |
| 3 | `optimizer` | 100, 100 | Los tres optimizadores con la tasa recibida |
| 4 | `get_model` | 100, 100 | Una neurona con sigmoide, compilada |
| 5 | `train_model` | 100, 100 | `model.fit`, que reproduce la pérdida esperada |
| 6 | `model_predict` | 100, 100 | Firma `model_predict(model, X_test)` y salida como tensor |

**Ideas para recordar**
1. Una regresión logística es una red de una neurona con sigmoide.
2. El desbalance se puede compensar dentro de la pérdida, sin quitar ni duplicar datos.
3. Las celdas de prueba dicen cómo se llamarán las funciones: dos firmas de la plantilla estaban mal.
4. Hay que evaluar en prueba: 98,8 % en entrenamiento frente a 91,7 % en prueba.

# **Créditos**
---

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Juan Sebastián Malagón Torres](https://sites.google.com/unal.edu.co/santiagotoledo-cortes/)
  * [Santiago Toledo Cortés](https://sites.google.com/unal.edu.co/santiagotoledo-cortes/)
* **Diseño de imágenes:**
    - [Mario Andres Rodriguez Triana](mailto:mrodrigueztr@unal.edu.co).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*